# Tutorial: Palette catalog summaries

Audience:
- Developers and designers who maintain a shared palette catalog (CSV) and want to
  understand how primary colors repeat across product types.

Prerequisites:
- Comfort reading Python.
- Familiarity with CSV files and basic data grouping (no external libraries needed).

Learning goals:
- By the end, the reader can write a small, dependency-free function that reads a
  CSV catalog, normalizes two text fields, groups rows by a normalized key, and
  returns a deterministically ordered summary.


## Outline

1. Setup
2. Define `summarize_palettes(csv_path)`
3. Run it on the catalog
4. Variations and pitfalls
5. Exercises


In [ ]:
# Setup cell: keep it short and deterministic
from __future__ import annotations

import csv
from collections import defaultdict

CSV_PATH = "/inputs/colors.csv"
CSV_PATH


## Step 1 - Define `summarize_palettes`

The function below uses only the Python standard library (`csv` and
`collections.defaultdict`). It:

- Reads the CSV using its header row (`csv.DictReader`), so column order in the
  file does not matter.
- Strips surrounding whitespace from `Primary (Hex)` and `Product Type`.
- Uppercases the stripped primary color to build the grouping key, so `#abc123`
  and `#ABC123` land in the same group.
- Counts every row in a group, including duplicate rows.
- Collects the sorted, unique, stripped product-type texts for each group.
- Returns groups ordered by count descending, then by primary color ascending
  (plain Python lexicographic string sorting).
- Returns an empty list for a header-only (otherwise empty) catalog.


In [ ]:
def summarize_palettes(csv_path: str) -> list[dict]:
    """Group a palette catalog CSV by normalized primary color.

    Each returned object has:
      - "primary": the stripped, uppercased primary color text.
      - "count": number of rows (including duplicates) sharing that primary color.
      - "product_types": sorted unique stripped product-type texts for that group.

    Groups are ordered by count descending, then primary ascending.
    """
    groups: dict[str, dict] = defaultdict(lambda: {"count": 0, "product_types": set()})

    with open(csv_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            primary_raw = row.get("Primary (Hex)", "")
            product_type_raw = row.get("Product Type", "")

            primary_raw = "" if primary_raw is None else primary_raw
            product_type_raw = "" if product_type_raw is None else product_type_raw

            primary = primary_raw.strip().upper()
            product_type = product_type_raw.strip()

            bucket = groups[primary]
            bucket["count"] += 1
            bucket["product_types"].add(product_type)

    summaries = [
        {
            "primary": primary,
            "count": bucket["count"],
            "product_types": sorted(bucket["product_types"]),
        }
        for primary, bucket in groups.items()
    ]

    summaries.sort(key=lambda item: (-item["count"], item["primary"]))
    return summaries


## Step 2 - Run it on the catalog

Call `summarize_palettes` on the real catalog file and inspect the top of the
result. The busiest primary color should sort first; ties break alphabetically
by the normalized primary color text.


In [ ]:
palette_summary = summarize_palettes('/inputs/colors.csv')
len(palette_summary), palette_summary[:3]


## Variations and pitfalls

- **Reordered columns**: because the function reads by header name via
  `csv.DictReader`, a schema-compatible file with columns in a different order
  still groups correctly.
- **Duplicate rows**: duplicates still count toward `count`, but contribute only
  one entry to `product_types` (a `set` de-duplicates automatically).
- **Case differences**: `#abc123` and `#ABC123` are normalized to the same
  group key because the primary color is uppercased after stripping.
- **Quoted product names with commas**: `csv.DictReader` understands RFC-4180
  quoting, so a product type like `"Tools, Analytics"` is parsed as one field,
  not split on the embedded comma.
- **Empty catalog**: a CSV with only a header row (no data rows) yields an
  empty `groups` dict and therefore an empty list â€” no special-casing needed.


## Exercises

1. Modify `summarize_palettes` to also report, per group, the single most
   common product type (ties broken alphabetically).
2. Write a tiny in-memory CSV string (using `io.StringIO`) with duplicate rows
   and a quoted product type containing a comma, and confirm the counts and
   `product_types` match your expectations.
3. Predict what `summarize_palettes` returns for a header-only CSV before
   running it, then check your prediction.


In [ ]:
# Exercise scaffold: try it on a tiny in-memory catalog
import io

def summarize_palettes_text(csv_text: str) -> list[dict]:
    """Helper for experimenting with small CSV snippets without writing files."""
    groups: dict[str, dict] = defaultdict(lambda: {"count": 0, "product_types": set()})
    reader = csv.DictReader(io.StringIO(csv_text))
    for row in reader:
        primary = (row.get("Primary (Hex)") or "").strip().upper()
        product_type = (row.get("Product Type") or "").strip()
        bucket = groups[primary]
        bucket["count"] += 1
        bucket["product_types"].add(product_type)
    summaries = [
        {"primary": p, "count": b["count"], "product_types": sorted(b["product_types"])}
        for p, b in groups.items()
    ]
    summaries.sort(key=lambda item: (-item["count"], item["primary"]))
    return summaries

sample_csv = (
    "Primary (Hex),Product Type\n"
    "  #abc123 , Tools \n"
    "#ABC123,\"Tools, Analytics\"\n"
    "#ABC123,Tools\n"
)
summarize_palettes_text(sample_csv)
